# S6E9：CatBoost 数值类别与数字拆分实验

目标：在原有 CatBoost 五折 baseline 上，检验数值的另一种表示是否提升 ROC AUC。

保留原始 13 个特征，新增数字拆分、收入尾数，以及 7 个数值特征的类别副本。
这些新列只由该行的特征生成；没有在全量训练集上计算目标均值。CatBoost 的类别统计由每折模型从该折训练数据学习。

**验证边界：本次研究先在原有第 1 折进行特征筛选。单折结果不是五折 OOF，也不是 Kaggle 成绩。此 notebook 的正式五折配置需完整运行后才能获得新版 OOF 与 submission。**

来源与思路：
- [比赛与评分规则](https://www.kaggle.com/competitions/playground-series-s6e9/overview)
- [参赛者的数字拆分消融实验](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/741755)
- [参赛者对重复收入值与编码信号的分析](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/742385)

以上公开实验提供待验证的假设；本 notebook 是独立实现，效果以自己的固定划分实验为准。

## 已完成的单折实验

下表为相同第 1 折、相同 2000 轮上限和 early stopping 设置的 ROC AUC。
基线来自用户已完成的原始训练日志；新特征模型在本地重新训练得到。

| 特征方案 | 第 1 折 AUC | 相比基线 |
|---|---:|---:|
| Original 13 features (user training log) | 0.94052506 | +0.00000000 |
| digits | 0.94279756 | +0.00227250 |
| digits_numeric_cats | 0.94468058 | +0.00415552 |

选中的方案是 **digits_numeric_cats**。`submission_numeric_features_fold1.csv` 由该单折模型预测完整的 286,571 条测试记录。
它是试验候选，**不是五折集成，尚无官方评分**。
该验证折同时用于 early stopping 和两种特征方案的筛选，仍需在其余四折验证提升能否保持。

此 notebook 的所有代码单元已按顺序通过 2,500 行训练数据、500 行测试数据、五折每折 3 轮的运行检查。
正式五折 2000 轮配置尚未执行。本地 HTML 预览受浏览器安全策略限制，未声称完成视觉检查。


## 1. 参数与依赖

使用现有 `dl` 内核即可。正式运行需要 pandas、numpy、scikit-learn、catboost。
随机划分与原 notebook 一致：`StratifiedKFold(5, shuffle=True, random_state=42)`。
默认输出到原项目的 `outputs/catboost_numeric_cats_v2` 子目录。

In [ ]:
from pathlib import Path
import json
import importlib.metadata
import time

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path('/Users/yw/unsw/Kaggle')
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'catboost_numeric_cats_v2'
MODE = 'digits_numeric_cats'  # 可改为 'raw' 或 'digits' 做同折对照
N_SPLITS = 5
SEED = 42
ITERATIONS = 2000
THREAD_COUNT = -1
EARLY_STOPPING = 150
SAMPLE_ROWS = None  # 正式评分保持 None；小样本只能用于检查代码
SAVE_MODELS = True

print({pkg: importlib.metadata.version(pkg)
       for pkg in ['pandas', 'numpy', 'scikit-learn', 'catboost']})

## 2. 读取与校验数据

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
target = 'Will_Buy_EV'

assert target not in test.columns
assert train['id'].is_unique and test['id'].is_unique
assert not train['id'].isin(test['id']).any()
assert set(train[target].unique()) == {'No', 'Yes'}
assert train.drop(columns=target).columns.tolist() == test.columns.tolist()
assert not train.isna().any().any() and not test.isna().any().any()

if SAMPLE_ROWS is not None:
    train = train.sample(SAMPLE_ROWS, random_state=SEED).reset_index(drop=True)
    test = test.head(min(500, len(test))).copy()

y = train[target].map({'No': 0, 'Yes': 1}).astype('int8')
print('train:', train.shape, 'test:', test.shape)
print('Yes rate:', y.mean())

## 3. 生成特征

例如，收入 `92887` 同时表示为连续数值、各位数字、尾部两/三位数字，以及类别 `"92887"`。
连续值帮助学习收入区间；类别副本让模型可以学习重复精确收入值的额外信号。
通勤距离先乘以 10 并四舍五入，再提取数字，避免二进制浮点数截断错误。

In [ ]:
NUMERIC = ['Age', 'Annual_Income_USD', 'Daily_Commute_km',
           'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
           'Charging_Stations_Near_Work', 'Environmental_Concern_Level']

def make_features(frame, mode):
    x = frame.drop(columns=['id', 'Will_Buy_EV'], errors='ignore').copy()
    if mode in ('digits', 'digits_numeric_cats'):
        for col, scale, positions in [('Age', 1, 2), ('Annual_Income_USD', 1, 6),
                                     ('Daily_Commute_km', 10, 3),
                                     ('Charging_Stations_Near_Home', 1, 2),
                                     ('Charging_Stations_Near_Work', 1, 2)]:
            values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
            for digit in range(positions):
                x[f'{col}_digit_{digit}'] = (values // (10 ** digit) % 10).astype(np.int8)
        income = np.rint(frame['Annual_Income_USD']).astype(np.int64)
        x['income_last_2'] = income % 100
        x['income_last_3'] = income % 1000
    if mode == 'digits_numeric_cats':
        for col in NUMERIC:
            scale = 10 if col == 'Daily_Commute_km' else 1
            x[f'{col}_as_category'] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    return x

In [ ]:
assert MODE in {'raw', 'digits', 'digits_numeric_cats'}
for frame in (train, test):
    assert np.allclose(frame['Annual_Income_USD'], frame['Annual_Income_USD'].round())
    commute_scaled = frame['Daily_Commute_km'] * 10
    assert np.allclose(commute_scaled, commute_scaled.round())

X = make_features(train, MODE)
X_test = make_features(test, MODE)
assert X.columns.tolist() == X_test.columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
print('Features:', X.shape[1], 'Categorical:', len(categorical_features))
print(categorical_features)

## 4. 五折训练与保存

每折只用训练索引拟合，以该折验证集做 early stopping；保留每折模型、验证预测和测试预测。
每条训练样本恰好产生一个 OOF 预测。五折完成后再生成最终平均概率。
`SAMPLE_ROWS` 有值时输出到 `smoke_test` 子目录，防止测试结果混入正式提交。

In [ ]:
run_dir = OUTPUT_DIR / MODE
if SAMPLE_ROWS is not None:
    run_dir = run_dir / 'smoke_test'
run_dir.mkdir(parents=True, exist_ok=True)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_pred = np.full(len(X), np.nan)
oof_coverage = np.zeros(len(X), dtype=np.int8)
test_predictions = []
fold_results = []
importance_values = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f'\n========== Fold {fold + 1} ==========')
    started = time.time()
    model = CatBoostClassifier(
        iterations=ITERATIONS, learning_rate=0.03, depth=7,
        loss_function='Logloss', eval_metric='AUC',
        random_seed=SEED + fold, verbose=400,
        thread_count=THREAD_COUNT, allow_writing_files=False,
    )
    model.fit(
        X.iloc[train_idx], y.iloc[train_idx],
        cat_features=categorical_features,
        eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
        early_stopping_rounds=EARLY_STOPPING,
        use_best_model=True,
    )
    assert np.array_equal(model.classes_, [0, 1])
    valid_pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
    fold_test_pred = model.predict_proba(X_test)[:, 1]
    oof_pred[valid_idx] = valid_pred
    oof_coverage[valid_idx] += 1
    test_predictions.append(fold_test_pred)
    importance_values.append(model.feature_importances_)
    row = {
        'fold': fold + 1,
        'auc': float(roc_auc_score(y.iloc[valid_idx], valid_pred)),
        'best_iteration': int(model.get_best_iteration()),
        'elapsed_seconds': time.time() - started,
    }
    fold_results.append(row)
    print(row)
    pd.DataFrame({'id': train['id'].iloc[valid_idx].to_numpy(),
                  'target': y.iloc[valid_idx].to_numpy(),
                  'prediction': valid_pred}).to_csv(run_dir / f'valid_fold_{fold + 1}.csv', index=False)
    pd.DataFrame({'id': test['id'].to_numpy(), 'Will_Buy_EV': fold_test_pred}).to_csv(
        run_dir / f'test_fold_{fold + 1}.csv', index=False)
    if SAVE_MODELS:
        model.save_model(str(run_dir / f'model_fold_{fold + 1}.cbm'))

assert len(fold_results) == N_SPLITS
assert (oof_coverage == 1).all() and np.isfinite(oof_pred).all()
test_pred = np.mean(test_predictions, axis=0)
assert len(test_pred) == len(test)
assert np.isfinite(test_pred).all() and ((test_pred >= 0) & (test_pred <= 1)).all()

metrics = {
    'mode': MODE, 'n_splits': N_SPLITS, 'seed': SEED,
    'iterations': ITERATIONS, 'sample_rows': SAMPLE_ROWS,
    'folds': fold_results,
    'mean_fold_auc': float(np.mean([r['auc'] for r in fold_results])),
    'oof_auc': float(roc_auc_score(y, oof_pred)),
    'feature_names': X.columns.tolist(),
}
(run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Mean Fold AUC:', metrics['mean_fold_auc'])
print('OOF AUC:', metrics['oof_auc'])

## 5. 导出 OOF、特征重要性和 submission

In [ ]:
pd.DataFrame({'id': train['id'].to_numpy(), 'target': y.to_numpy(),
              'prediction': oof_pred}).to_csv(run_dir / 'oof.csv', index=False)
importance = pd.DataFrame({'feature': X.columns,
                           'importance': np.mean(importance_values, axis=0)}).sort_values(
                               'importance', ascending=False)
importance.to_csv(run_dir / 'feature_importance.csv', index=False)
submission = pd.DataFrame({'id': test['id'].to_numpy(), 'Will_Buy_EV': test_pred})
name = 'submission.csv' if SAMPLE_ROWS is None else 'smoke_predictions_DO_NOT_SUBMIT.csv'
submission.to_csv(run_dir / name, index=False)
print('Saved:', run_dir / name)
display(pd.DataFrame(fold_results))
display(importance.head(12))
display(submission.head())

## 6. 如何判断是否有效

原始模型的五折 AUC（来自原 notebook 的已完成训练日志）：
`[0.9405250587, 0.9413214882, 0.9427254533, 0.9421698643, 0.9417917527]`。
原始 OOF 为 **0.9416996135**，用户报告最新 Public LB 为 **0.94149**。

正式运行后，逐折比较新版与原版 AUC，并检查整体 OOF。若增益只出现在用于筛选的第 1 折，不能据此宣称稳定提升。
本地 OOF 与 Public LB 不来自同一批样本。只有上传新预测并收到官方评分后，才能报告新的榜单成绩。
模型文件、OOF 和测试预测已保存，后续融合应使用同一验证划分和对齐的 id。